# Hyperparameters Search

This notebook runs the full hyperparameter-search and visualization workflow for multiple synthetic dataset sizes:

`dataset_sizes = [1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000, 256000]`

**1. Symbolic Physics Generation**
Uses `SymPy` to derive the symbolic Gibbs-Appell equations for a 2-link robot arm. These are lambdified into fast `NumPy` functions to serve as the ground truth for the physics engine.

**2. Data Pipeline**
Generates synthetic state configurations ($q, \dot{q}, \ddot{q}$) and calculates the corresponding torque ($	au$) and energy scalars ($S$). Data are normalized and split into train/test sets with pre-calculated chain-rule scaling factors for each dataset size.

**3. Optuna Hyperparameter Search**
Instead of a fixed architecture, the model is defined dynamically. Optuna sweeps through a search space to optimize:
* **Structure:** Network depth and width per layer.
* **Hyperparameters:** Learning rate and activation functions (ReLU, GELU, Tanh, etc.).

**4. Physics-Informed Loss**
The model predicts the scalar energy $S$. We use `torch.autograd` to differentiate the output with respect to acceleration ($\ddot{q}$) to obtain torque: $	au = \partial S / \partial \ddot{q}$. The loss function minimizes the error on **torque**, forcing the network to learn the underlying physics derivatives.

**5. Analysis & Visualization**
For every dataset size, the notebook saves Optuna artifacts and generates the existing visualizations with size-specific labels:
* **Performance:** RMSE comparison across different activation functions.
* **Sensitivity:** fANOVA analysis to rank hyperparameter importance.
* **Landscapes:** 2D scatter plots of loss vs. model complexity.


In [9]:
%pip install sympy numpy torch matplotlib seaborn optuna joblib pandas plotly scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
import sympy as sp
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
import optuna
import joblib
import json
import pandas as pd
from optuna.visualization import plot_parallel_coordinate, plot_param_importances, plot_optimization_history
from optuna.importance import get_param_importances

In [11]:
# Configuration
SEED = 22
BATCH_SIZE = 64
N_TRIALS = 500
dataset_sizes = [16000]
HYPERPARAMETER_OUTPUT_DIR = "hyperparameter_search_by_size"
PARAMS = {'m1': 1.0, 'm2': 1.0, 'l1': 1.0, 'l2': 1.0}

os.makedirs(HYPERPARAMETER_OUTPUT_DIR, exist_ok=True)

print("Libraries imported and seed set.")


Libraries imported and seed set.


In [12]:
# 1. Define Time and Parameters
t = sp.symbols('t')
m1, m2, l1, l2, g = sp.symbols('m1 m2 l1 l2 g')
theta1 = sp.Function('theta1')(t)
theta2 = sp.Function('theta2')(t)

# 2. Shorthand for derivatives
q1, q2 = sp.symbols('q1 q2')
dq1, dq2 = sp.symbols('dq1 dq2')
ddq1, ddq2 = sp.symbols('ddq1 ddq2')

# 3. Kinematics
x1 = l1 * sp.sin(theta1)
y1 = -l1 * sp.cos(theta1)
x2 = x1 + l2 * sp.sin(theta1 + theta2)
y2 = y1 - l2 * sp.cos(theta1 + theta2)

# Velocities
vx1, vy1 = sp.diff(x1, t), sp.diff(y1, t)
vx2, vy2 = sp.diff(x2, t), sp.diff(y2, t)

# Accelerations
ax1, ay1 = sp.diff(vx1, t), sp.diff(vy1, t)
ax2, ay2 = sp.diff(vx2, t), sp.diff(vy2, t)

# 4. Gibbs-Appell Function S
accel_mag_sq_1 = ax1**2 + ay1**2
accel_mag_sq_2 = ax2**2 + ay2**2
S_symbolic = 0.5 * m1 * accel_mag_sq_1 + 0.5 * m2 * accel_mag_sq_2

# Substitution map to clean symbols
subs_map = {
    theta1.diff(t, t): ddq1, theta2.diff(t, t): ddq2,
    theta1.diff(t): dq1, theta2.diff(t): dq2,
    theta1: q1, theta2: q2
}
S_clean = S_symbolic.subs(subs_map).simplify()

# Forces (Derivatives of S w.r.t acceleration)
tau1_sym = sp.diff(S_clean, ddq1)
tau2_sym = sp.diff(S_clean, ddq2)

# Create Fast Lambdified Functions
compute_S_ground_truth = sp.lambdify(
    (q1, q2, dq1, dq2, ddq1, ddq2, m1, m2, l1, l2), S_clean, 'numpy'
)
compute_tau_ground_truth = sp.lambdify(
    (q1, q2, dq1, dq2, ddq1, ddq2, m1, m2, l1, l2), [tau1_sym, tau2_sym], 'numpy'
)

print("Physics engine initialized.")

Physics engine initialized.


In [13]:
def seed_everything(seed=27):
    # 1. Python standard library
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 2. NumPy (Data generation)
    np.random.seed(seed)

    # 3. PyTorch (Model weights & shuffling)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # For multi-GPU

    # 4. Force deterministic algorithms (Crucial for reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [14]:
def generate_data(n_samples):
    q = np.random.uniform(-np.pi/2, np.pi/2, (n_samples, 2))
    dq = np.random.uniform(-5.0, 5.0, (n_samples, 2))
    ddq = np.random.uniform(-20.0, 20.0, (n_samples, 2))

    # Compute S
    S_values = compute_S_ground_truth(
        q[:, 0], q[:, 1], dq[:, 0], dq[:, 1], ddq[:, 0], ddq[:, 1],
        PARAMS['m1'], PARAMS['m2'], PARAMS['l1'], PARAMS['l2']
    )

    # Compute Forces
    tau_values = compute_tau_ground_truth(
        q[:, 0], q[:, 1], dq[:, 0], dq[:, 1], ddq[:, 0], ddq[:, 1],
        PARAMS['m1'], PARAMS['m2'], PARAMS['l1'], PARAMS['l2']
    )
    tau_values = np.array(tau_values).T

    X = np.hstack([q, dq, ddq])
    return X.astype(np.float32), S_values.reshape(-1, 1).astype(np.float32), tau_values.astype(np.float32)


def prepare_dataset(dataset_size):
    seed_everything(seed=SEED)

    # Generate Data
    X_full, S_full, tau_full = generate_data(dataset_size)

    # Normalization & Scaling
    X_mean = X_full.mean(axis=0)
    X_std = X_full.std(axis=0)
    S_mean = S_full.mean()
    S_std = S_full.std()

    X_norm = (X_full - X_mean) / (X_std + 1e-8)
    S_norm = (S_full - S_mean) / (S_std + 1e-8)

    # Chain Rule Scaling Factors
    ddq_std = X_std[4:6]
    chain_rule_scale = S_std / (ddq_std + 1e-8)

    # Split Train/Test
    split_idx = int(dataset_size * 0.8)

    return {
        "dataset_size": dataset_size,
        "X_full": X_full,
        "S_full": S_full,
        "tau_full": tau_full,
        "X_mean": X_mean,
        "X_std": X_std,
        "S_mean": S_mean,
        "S_std": S_std,
        "X_norm": X_norm,
        "S_norm": S_norm,
        "chain_rule_scale": chain_rule_scale,
        "X_train": torch.tensor(X_norm[:split_idx]),
        "tau_train": torch.tensor(tau_full[:split_idx]),
        "X_test": torch.tensor(X_norm[split_idx:]),
        "tau_test": torch.tensor(tau_full[split_idx:]),
    }


print(f"Data preparation ready for dataset sizes: {dataset_sizes}")


Data preparation ready for dataset sizes: [16000]


In [15]:
# 1. Dynamic Model Builder (Optuna-driven)
def build_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 10)

    # Select the activation function for this trial
    activation_name = trial.suggest_categorical("activation", ["GELU", "Tanh", "Softplus", "ELU", "ReLU"])

    layers = []
    in_features = 6 # [q, dq, ddq]

    for i in range(n_layers):
        out_features = trial.suggest_int(f"n_units_l{i}", 1, 256)
        layers.append(nn.Linear(in_features, out_features))

        # Instantiate the chosen activation class dynamically
        layers.append(getattr(nn, activation_name)())

        #p = trial.suggest_float(f"dropout_l{i}", 0.0, 0.3)
        #layers.append(nn.Dropout(p))

        in_features = out_features

    layers.append(nn.Linear(in_features, 1)) # Output S
    return nn.Sequential(*layers)

# 2. Physics Loss Function (Unchanged, but necessary for context)
def compute_force_loss_chain_rule(model, X_batch, tau_batch_real, chain_scale):
    # Enable grad for input derivative (dS/dq)
    X_batch_grad = X_batch.clone().requires_grad_(True)

    S_pred_norm = model(X_batch_grad)
    S_sum = S_pred_norm.sum()

    # Calculate gradient of S w.r.t input
    grads_norm = autograd.grad(S_sum, X_batch_grad, create_graph=True)[0]

    dS_dddq_norm = grads_norm[:, 4:6]

    if not isinstance(chain_scale, torch.Tensor):
        chain_scale = torch.tensor(chain_scale, dtype=torch.float32, device=X_batch.device)

    tau_pred_real = dS_dddq_norm * chain_scale
    loss = torch.mean((tau_pred_real - tau_batch_real) ** 2)
    return loss

# 3. Optimization Loop
def objective(trial):
    model = build_model(trial)

    lr = trial.suggest_float("lr", 1e-8, 1e-1, log=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    SEARCH_EPOCHS = 100

    for epoch in range(SEARCH_EPOCHS):
        model.train()
        indices = torch.randperm(X_train.shape[0])

        for start_idx in range(0, X_train.shape[0], BATCH_SIZE):
            end_idx = min(start_idx + BATCH_SIZE, X_train.shape[0])
            batch_indices = indices[start_idx:end_idx]

            X_batch = X_train[batch_indices]
            tau_batch = tau_train[batch_indices]

            optimizer.zero_grad()
            loss = compute_force_loss_chain_rule(model, X_batch, tau_batch, chain_rule_scale)
            loss.backward()
            optimizer.step()

        # --- Validation ---
        model.eval()
        # No 'with torch.no_grad()' because we need derivatives for the loss!
        val_loss = compute_force_loss_chain_rule(model, X_test, tau_test, chain_rule_scale)

        trial.report(val_loss.item(), epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_loss.item()

In [ ]:
# 1. Run one complete Optuna study for each dataset size
study_results = {}

for dataset_size in dataset_sizes:
    print()
    print("="*72)
    print(f"Starting optimization for dataset_size={dataset_size:,} with fixed seed {SEED}...")
    print("="*72)

    dataset_context = prepare_dataset(dataset_size)

    # The objective function reads these globals during each Optuna trial.
    X_train = dataset_context["X_train"]
    tau_train = dataset_context["tau_train"]
    X_test = dataset_context["X_test"]
    tau_test = dataset_context["tau_test"]
    chain_rule_scale = dataset_context["chain_rule_scale"]

    # 2. Setup the Sampler AND Pruner
    seed_everything(SEED)
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=10)

    # 3. Create Study with Pruner
    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        study_name=f"dataset_size_{dataset_size}"
    )

    study.optimize(objective, n_trials=N_TRIALS)

    study_results[dataset_size] = {
        "study": study,
        "dataset_context": dataset_context,
    }

print()
print("Completed optimization for all dataset sizes.")


[I 2026-04-20 20:08:37,372] A new study created in memory with name: dataset_size_16000



Starting optimization for dataset_size=16,000 with fixed seed 22...


[I 2026-04-20 20:08:47,639] Trial 0 finished with value: 0.12605136632919312 and parameters: {'n_layers': 3, 'activation': 'Softplus', 'n_units_l0': 70, 'n_units_l1': 177, 'n_units_l2': 57, 'lr': 0.004826768210503866}. Best is trial 0 with value: 0.12605136632919312.
[I 2026-04-20 20:08:51,676] Trial 1 finished with value: 17.94842529296875 and parameters: {'n_layers': 1, 'activation': 'Tanh', 'n_units_l0': 198, 'lr': 0.050678666245081064}. Best is trial 0 with value: 0.12605136632919312.
[I 2026-04-20 20:09:21,315] Trial 2 finished with value: 1763.9267578125 and parameters: {'n_layers': 8, 'activation': 'Tanh', 'n_units_l0': 110, 'n_units_l1': 150, 'n_units_l2': 180, 'n_units_l3': 29, 'n_units_l4': 237, 'n_units_l5': 254, 'n_units_l6': 174, 'n_units_l7': 204, 'lr': 1.5978442544739387e-08}. Best is trial 0 with value: 0.12605136632919312.


In [ ]:
for dataset_size, result in study_results.items():
    study = result["study"]
    pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

    print()
    print("="*40)
    print(f"Study statistics for dataset_size={dataset_size:,}:")
    print(f"  Total Trials:     {len(study.trials)}")
    print(f"  Complete Trials:  {len(complete_trials)}")
    print(f"  Pruned Trials:    {len(pruned_trials)}")
    print("="*40)


In [ ]:
# --- RESULTS & SAVING ---
print()
print("="*40)
print("OPTIMIZATION FINISHED")
print("="*40)

best_hyperparameters_by_size = {}
trial_history_frames = []

for dataset_size, result in study_results.items():
    study = result["study"]

    print()
    print("-"*40)
    print(f"Results for dataset_size={dataset_size:,}")
    print("-"*40)

    # A. Print Best Results
    best_mse = study.best_trial.value
    best_rmse = np.sqrt(best_mse)
    print(f"Best MSE Loss: {best_mse:.6f}")
    print(f"Best RMSE:     {best_rmse:.6f} (N*m)")

    print("Best Params:")
    for key, value in study.best_trial.params.items():
        print(f"  {key}: {value}")

    # B. Save the Study History (CSV)
    df_results = study.trials_dataframe()
    df_results.insert(0, "dataset_size", dataset_size)

    # Optuna names the loss column "value" by default
    df_results["RMSE"] = np.sqrt(df_results["value"])

    csv_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"optuna_search_results_{dataset_size}.csv")
    df_results.to_csv(csv_path, index=False)
    trial_history_frames.append(df_results)
    print(f"[Saved] Full trial history (with RMSE) saved to '{csv_path}'")

    # C. Save the Study Object (PKL)
    study_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"optuna_study_{dataset_size}.pkl")
    joblib.dump(study, study_path)
    print(f"[Saved] Study object saved to '{study_path}'")

    # D. Save Best Parameters (JSON)
    best_payload = {
        "dataset_size": dataset_size,
        "best_mse": best_mse,
        "best_rmse": best_rmse,
        "params": study.best_trial.params,
    }
    best_hyperparameters_by_size[str(dataset_size)] = best_payload

    best_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"best_hyperparameters_{dataset_size}.json")
    with open(best_path, "w") as f:
        json.dump(best_payload, f, indent=4)
    print(f"[Saved] Best hyperparameters saved to '{best_path}'")

combined_csv_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, "optuna_search_results_all_sizes.csv")
pd.concat(trial_history_frames, ignore_index=True).to_csv(combined_csv_path, index=False)
print()
print(f"[Saved] Combined trial history saved to '{combined_csv_path}'")

combined_best_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, "best_hyperparameters_by_size.json")
with open(combined_best_path, "w") as f:
    json.dump(best_hyperparameters_by_size, f, indent=4)
print(f"[Saved] Combined best hyperparameters saved to '{combined_best_path}'")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- CONSTANTS ---
FIG_SIZE = (3.6, 3.2)

# --- 1. CONFIGURATION ---
plt.rcParams['svg.fonttype'] = 'none'

for dataset_size in dataset_sizes:
    # Load data
    csv_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"optuna_search_results_{dataset_size}.csv")
    df = pd.read_csv(csv_path)

    # Initialize the figure canvas
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    # --- 2. DATA PREPARATION ---
    df_plot = df.copy()
    df_plot.rename(columns={"params_activation": "Activation", "value": "Loss"}, inplace=True)

    if "RMSE" not in df_plot.columns:
        df_plot["RMSE"] = np.sqrt(df_plot["Loss"])

    order = ["ReLU", "GELU", "ELU", "Softplus", "Tanh"]

    # Dictionary mapping each activation to the requested shapes
    markers = {"ReLU": "o", "GELU": "D", "ELU": "s", "Softplus": "X", "Tanh": "P"}

    # Extract a single row dataframe containing ONLY the absolute best configuration
    best_idx = df_plot["RMSE"].idxmin()
    df_best = df_plot.loc[[best_idx]]
    optimal_activation = df_best["Activation"].values[0]

    # --- 3. LAYER 1: THE RAW DATA (BACKGROUND) ---
    # Loop through each category to apply the unique shapes
    for act in order:
        subset = df_plot[df_plot["Activation"] == act]
        sns.stripplot(
            x="Activation",
            y="RMSE",
            data=subset,
            order=order,
            color="#EEEEEE",
            marker=markers[act],
            size=3,
            jitter=0.2,
            alpha=0.6,
            zorder=0,
            legend=False,
            ax=ax
        )

    # --- 4. LAYER 2: THE OPTIMAL POINT (FOREGROUND) ---
    # The single optimal point is pure red and matches its category shape
    sns.stripplot(
        x="Activation",
        y="RMSE",
        data=df_best,
        order=order,
        color="red",
        marker=markers[optimal_activation],
        size=4,
        jitter=0,
        alpha=1.0,
        zorder=20,
        legend=False,
        ax=ax
    )

    # --- 5. LAYER 3: THE STATISTICS (FOREGROUND) ---
    point_palette = {act: "#757575" for act in order}
    marker_list = [markers[act] for act in order]

    sns.pointplot(
        x="Activation",
        y="RMSE",
        hue="Activation",
        data=df_plot,
        order=order,
        hue_order=order,
        palette=point_palette,
        markers=marker_list,
        errorbar='sd',
        capsize=0,
        linestyle='none',
        markersize=4,
        err_kws={'linewidth': 1.2},
        zorder=10,
        legend=False,
        ax=ax
    )

    # --- 6. CLEANUP & AESTHETICS ---
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_title(f"Activation Search (n={dataset_size:,})", loc='left', pad=10, fontweight='bold', fontsize=11)
    ax.set_ylabel("Torque RMSE (N m)")
    ax.set_xlabel("")

    ax.set_yscale("linear")
    ax.set_ylim(0, 50)

    plt.tight_layout()
    figure_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"figure_a_modern_linear_fixed_{dataset_size}.svg")
    plt.savefig(figure_path, format="svg")
    print(f"[Saved] Activation search plot for dataset_size={dataset_size:,} to '{figure_path}'")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import joblib
import optuna
from optuna.importance import get_param_importances, FanovaImportanceEvaluator

# 1. Config & Setup
plt.rcParams['svg.fonttype'] = 'none'

label_map = {
    "params_lr":          "Learning Rate",
    "params_n_layers":    "Network Depth",
    "params_hidden_dim":  "Layer Width",
    "params_activation":  "Activation Function",
    "params_dropout":     "Dropout Rate",
    "params_optimizer":   "Optimizer Type"
}

for dataset_size in dataset_sizes:
    # --- LOAD THE STUDY FOR THIS DATASET SIZE ---
    if "study_results" in globals() and dataset_size in study_results:
        study = study_results[dataset_size]["study"]
    else:
        study_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"optuna_study_{dataset_size}.pkl")
        study = joblib.load(study_path)

    # 2. Calculate Importance (Stable)
    # The evaluator uses the seed to ensure numbers don't wiggle
    evaluator = FanovaImportanceEvaluator(seed=SEED)
    importances = get_param_importances(study, evaluator=evaluator)

    # 3. Setup Figure
    fig, ax = plt.subplots(figsize=(4.4, 2.7))

    clean_labels = [label_map.get(k, k) for k in importances.keys()]
    values = list(importances.values())

    # 4. Sort (Big bar at top)
    sorted_indices = np.argsort(values)
    clean_labels = [clean_labels[i] for i in sorted_indices]
    values = [values[i] for i in sorted_indices]

    # 5. Plot
    bars = ax.barh(clean_labels, values, color="#666666", height=0.6)

    # 6. Add Percentages
    for bar in bars:
        width = bar.get_width()
        ax.text(
            width + 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{width:.1%}',
            va='center',
            fontsize=10,
            color="#333333"
        )

    # 7. Cleanup
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    ax.set_xticks([])
    ax.set_title(f"Hyperparameter Sensitivity (n={dataset_size:,})", loc='left', pad=10, fontweight='bold', fontsize=11)

    plt.tight_layout()
    figure_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"figure_b_sensitivity_stable_{dataset_size}.svg")
    plt.savefig(figure_path, format="svg")
    print(f"[Saved] Sensitivity plot for dataset_size={dataset_size:,} to '{figure_path}'")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- CONSTANTS ---
FIG_SIZE = (12, 8)

# --- 1. CONFIGURATION ---
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

for dataset_size in dataset_sizes:
    # Load Data
    csv_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"optuna_search_results_{dataset_size}.csv")
    df = pd.read_csv(csv_path)

    # --- 2. DATA PREPARATION ---
    df_plot = df.copy()

    # Updated map: Added Layer 3
    param_map = {
        "params_lr":          "Learning Rate",
        "params_n_units_l0":  "# Comp. Units (Layer 0)",
        "params_n_units_l1":  "# Comp. Units (Layer 1)",
        "params_n_units_l2":  "# Comp. Units (Layer 2)",
        "params_n_units_l3":  "# Comp. Units (Layer 3)",
        "params_n_layers":    "Network Depth",
        "params_activation":  "Activation",
        "value":              "Loss"
    }
    df_plot.rename(columns=param_map, inplace=True)

    if "RMSE" not in df_plot.columns:
        df_plot["RMSE"] = np.sqrt(df_plot["Loss"])

    # Now we have 6 items to plot
    params_to_plot = [
        "Learning Rate", "Network Depth",
        "# Comp. Units (Layer 0)", "# Comp. Units (Layer 1)",
        "# Comp. Units (Layer 2)", "# Comp. Units (Layer 3)"
    ]

    # Extract a 1-row DataFrame containing ONLY the absolute best configuration
    best_idx = df_plot["RMSE"].idxmin()
    df_best = df_plot.loc[[best_idx]]

    # --- 3. PLOTTING LOOP ---
    # 2 rows, 3 columns = 6 slots (all used)
    fig, axes = plt.subplots(2, 3, figsize=FIG_SIZE, sharey=True)
    axes = axes.flatten()

    # Color & Shape Palette
    hue_order = ["ReLU", "GELU", "ELU", "Softplus", "Tanh"]
    markers = {"ReLU": "o", "GELU": "D", "ELU": "s", "Softplus": "X", "Tanh": "P"}

    # Palettes to separate the background (grey) and the optimal point (red)
    bg_palette = {act: "#BBBBBB" for act in hue_order}
    opt_palette = {act: "red" for act in hue_order}

    handles, labels = None, None

    for i in range(len(axes)):
        ax = axes[i]

        # Check if we have a parameter for this subplot slot
        if i < len(params_to_plot):
            param = params_to_plot[i]

            if param not in df_plot.columns:
                ax.set_visible(False)
                continue

            # LAYER 1: Background Scatter Plot (All points, greyed out)
            sns.scatterplot(
                data=df_plot,
                x=param,
                y="RMSE",
                hue="Activation",
                palette=bg_palette,
                hue_order=hue_order,
                style="Activation",
                markers=markers,
                s=50,
                alpha=0.8,
                edgecolor="white",
                linewidth=0.3,
                legend=(i == 0),
                ax=ax,
                zorder=1
            )

            if i == 0:
                handles, labels = ax.get_legend_handles_labels()
                ax.get_legend().remove()

            # LAYER 2: Foreground Scatter Plot (ONLY the optimal point, red)
            sns.scatterplot(
                data=df_best,
                x=param,
                y="RMSE",
                hue="Activation",
                palette=opt_palette,
                hue_order=hue_order,
                style="Activation",
                markers=markers,
                s=50,
                alpha=1.0,
                edgecolor="white",
                linewidth=0.3,
                legend=False,
                ax=ax,
                zorder=10
            )

            if "Learning Rate" in param:
                ax.set_xscale("log")

            # Aesthetics
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_xlabel(param, fontsize=10, fontweight='bold')

            # Y-Label Logic: Show on the first plot of each row (index 0 and 3)
            if i % 3 == 0:
                ax.set_ylabel("Torque RMSE (N m)", fontsize=10)
            else:
                ax.set_ylabel("")
                ax.tick_params(axis='y', which='both', left=False)

            ax.yaxis.grid(True, linestyle=':', alpha=0.4, color='grey')

        else:
            # Hide any truly unused subplot slots
            ax.set_visible(False)

    # --- 4. GLOBAL LEGEND ---
    if handles and labels:
        fig.legend(
            handles, labels,
            loc='upper center',
            bbox_to_anchor=(0.5, 0.94),
            ncol=5,
            frameon=False,
            fontsize=10
        )

    plt.suptitle(f"Hyperparameter Optimization Landscapes (n={dataset_size:,})", fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.90])

    figure_path = os.path.join(HYPERPARAMETER_OUTPUT_DIR, f"figure_c_grid_landscapes_fixed_{dataset_size}.svg")
    plt.savefig(figure_path, format="svg")
    print(f"[Saved] Landscape plot for dataset_size={dataset_size:,} to '{figure_path}'")
    plt.show()
